# 01.2 — Bronze ingest (Databricks)
Runs **inside Databricks**, after verified files have been uploaded to the Unity Catalog Volume by the unchanged local extractor. Uses explicit schemas, quarantines malformed/mismatched CSV lines, MERGEs idempotently, and logs each file. The default Volume root preserves the original deployed Databricks path; use the `volume_root` widget if the upload destination differs.

In [0]:
import re
import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, LongType, DoubleType, StringType, TimestampType
)
from delta.tables import DeltaTable

spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG = "workspace"
SCHEMA = "cryptolens"
BRONZE_KLINES_TABLE = f"{CATALOG}.{SCHEMA}.bronze_klines"
BRONZE_EXCHANGE_INFO_TABLE = f"{CATALOG}.{SCHEMA}.bronze_exchange_info"
BRONZE_QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_quarantine"
LOG_TABLE = f"{CATALOG}.{SCHEMA}.pipeline_execution_logs"

# Keep exactly the 12 Binance CSV fields, in source order. The CSV is
# headerless. Parse via from_csv on raw text, so extra fields can be detected
# rather than silently discarded by PERMISSIVE CSV parsing.
RAW_KLINE_SCHEMA = StructType([
    StructField("open_time", LongType(), True),
    StructField("open", DoubleType(), True),
    StructField("high", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("close", DoubleType(), True),
    StructField("volume", DoubleType(), True),
    StructField("close_time", LongType(), True),
    StructField("quote_asset_volume", DoubleType(), True),
    StructField("number_of_trades", LongType(), True),
    StructField("taker_buy_base_asset_volume", DoubleType(), True),
    StructField("taker_buy_quote_asset_volume", DoubleType(), True),
    StructField("ignore", StringType(), True),
])
RAW_TEXT_SCHEMA = StructType([StructField("value", StringType(), True)])
RAW_FIELDS = [f.name for f in RAW_KLINE_SCHEMA.fields]
REQUIRED_FIELDS = RAW_FIELDS[:-1]  # 'ignore' may be empty

LOG_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("parameter_processed", StringType(), False),
    StructField("load_type", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("rows_inserted", LongType(), True),
    StructField("rows_updated", LongType(), True),
    StructField("error_message", StringType(), True),
])

## Parameters
`year/month/day` select which files to ingest; this is not a downloader. Set `volume_root` to the exact existing Volume directory containing `full_load`, `incremental_load` and `exchange_info`. Do not rename or relocate local extractor files.

In [0]:
dbutils.widgets.text("volume_root", "/Volumes/workspace/cryptolens/bronze_raw/l242529@lhr.nu.edu.pk")
dbutils.widgets.text("interval", "1m")
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental"])
dbutils.widgets.text("year", "2026")
dbutils.widgets.text("month", "09")
dbutils.widgets.text("day", "25")
dbutils.widgets.text("snapshot_date", "")
dbutils.widgets.dropdown("require_files", "true", ["true", "false"])

VOLUME_ROOT = dbutils.widgets.get("volume_root").rstrip("/")
interval = dbutils.widgets.get("interval").strip()
load_type = dbutils.widgets.get("load_type")
year = int(dbutils.widgets.get("year"))
month = int(dbutils.widgets.get("month"))
day_str = dbutils.widgets.get("day").strip()
day = int(day_str) if day_str else None
snapshot_date = dbutils.widgets.get("snapshot_date").strip() or None
require_files = dbutils.widgets.get("require_files") == "true"

if not VOLUME_ROOT.startswith("/Volumes/"):
    raise ValueError("volume_root must be a Databricks Unity Catalog /Volumes/... path")
if not interval or not 1 <= month <= 12 or (load_type == "incremental" and (day is None or not 1 <= day <= 31)):
    raise ValueError("Check interval / month / day parameters")

In [0]:
def utc_now():
    return datetime.now(timezone.utc).replace(tzinfo=None)


def log_execution(layer, param, kind, start, status, inserted=0, updated=0, error=None):
    row = [(str(uuid.uuid4()), layer, param, kind, start, utc_now(), status,
            int(inserted), int(updated), str(error)[:1000] if error else None)]
    spark.createDataFrame(row, schema=LOG_SCHEMA).write.format("delta").mode("append").saveAsTable(LOG_TABLE)


def merge_counts(table):
    history = DeltaTable.forName(spark, table).history(1).select("operation", "operationMetrics").first()
    if history is None or history["operation"] != "MERGE":
        return 0, 0
    metrics = history["operationMetrics"] or {}
    return int(metrics.get("numTargetRowsInserted", 0)), int(metrics.get("numTargetRowsUpdated", 0))


def list_files(folder):
    try:
        return dbutils.fs.ls(folder)
    except Exception as exc:
        raise RuntimeError(f"Cannot access Databricks Volume folder {folder}: {exc}") from exc


def build_volume_path(symbol, interval, load_type, year, month, day=None):
    if load_type == "full":
        fname = f"{symbol}-{interval}-{year}-{month:02d}.csv"
        return f"{VOLUME_ROOT}/full_load/{fname}"
    if load_type == "incremental" and day is not None:
        fname = f"{symbol}-{interval}-{year}-{month:02d}-{day:02d}.csv"
        return f"{VOLUME_ROOT}/incremental_load/{fname}"
    raise ValueError("Incremental files require a day")


def discover_symbols(interval, load_type, year, month, day=None):
    folder = f"{VOLUME_ROOT}/{'full_load' if load_type == 'full' else 'incremental_load'}"
    suffix = (f"-{interval}-{year}-{month:02d}.csv" if load_type == "full"
              else f"-{interval}-{year}-{month:02d}-{day:02d}.csv")
    return sorted({file.name[:-len(suffix)] for file in list_files(folder)
                   if file.name.endswith(suffix) and re.fullmatch(r"[A-Z0-9]+", file.name[:-len(suffix)])})

## Raw → Bronze
Parse numeric source values using an explicit StructType. Raw CSV lines with an unexpected number of fields, failed numeric conversions or missing required values are stored in `bronze_quarantine` with their original line. A re-run with unchanged input writes no duplicate Bronze records and performs no matched updates.

In [0]:
def ingest_kline_file(symbol, interval, load_type, file_path):
    started = utc_now()
    try:
        raw = spark.read.schema(RAW_TEXT_SCHEMA).text(file_path).withColumnRenamed("value", "raw_line")
        parsed = (raw
            .withColumn("field_count", F.size(F.split(F.col("raw_line"), ",", -1)))
            .withColumn("parsed", F.from_csv(F.col("raw_line"), RAW_KLINE_SCHEMA,
                                              {"mode": "PERMISSIVE", "header": "false"})))
        for field in RAW_FIELDS:
            parsed = parsed.withColumn(field, F.col(f"parsed.`{field}`"))

        missing = None
        for field in REQUIRED_FIELDS:
            condition = F.col(field).isNull()
            missing = condition if missing is None else (missing | condition)
        bad_condition = (F.col("field_count") != F.lit(12)) | missing
        bad = parsed.filter(bad_condition)
        bad_count = bad.count()
        if bad_count:
            quarantined = (bad.select(
                    F.col("raw_line").alias("_corrupt_record"),
                    F.col("field_count"),
                    F.when(F.col("field_count") != 12, F.lit("UNEXPECTED_COLUMN_COUNT"))
                     .otherwise(F.lit("PARSE_OR_REQUIRED_FIELD_FAILURE")).alias("bad_reason"))
                .withColumn("symbol", F.lit(symbol))
                .withColumn("interval", F.lit(interval))
                .withColumn("load_type", F.lit(load_type))
                .withColumn("source_file", F.lit(file_path))
                .withColumn("load_timestamp", F.current_timestamp()))
            (quarantined.write.format("delta").mode("append")
             .option("mergeSchema", "true").saveAsTable(BRONZE_QUARANTINE_TABLE))

        good = (parsed.filter(~bad_condition).select(*RAW_FIELDS)
            .withColumn("symbol", F.lit(symbol))
            .withColumn("interval", F.lit(interval))
            .withColumn("load_type", F.lit(load_type))
            .withColumn("source_file", F.lit(file_path))
            .withColumn("ingestion_date", F.current_date())
            .withColumn("load_timestamp", F.current_timestamp()))

        rows = good.count()
        if not rows:
            raise ValueError(f"No valid CSV rows from {file_path} (quarantined={bad_count})")

        if not spark.catalog.tableExists(BRONZE_KLINES_TABLE):
            good.write.format("delta").saveAsTable(BRONZE_KLINES_TABLE)
            inserted, updated = rows, 0
        else:
            business = RAW_FIELDS
            changed = "NOT (" + " AND ".join(f"t.`{c}` <=> s.`{c}`" for c in business) + ")"
            target = DeltaTable.forName(spark, BRONZE_KLINES_TABLE)
            (target.alias("t").merge(good.alias("s"),
                "t.symbol = s.symbol AND t.interval = s.interval AND "
                "t.load_type = s.load_type AND t.open_time = s.open_time")
                .whenMatchedUpdateAll(condition=changed)
                .whenNotMatchedInsertAll().execute())
            inserted, updated = merge_counts(BRONZE_KLINES_TABLE)

        log_execution("Raw-to-Bronze", file_path, load_type, started, "SUCCESS", inserted, updated)
        print(f"[OK] {symbol}: valid={rows:,}, quarantined={bad_count:,}, inserted={inserted:,}, updated={updated:,}")
    except Exception as exc:
        log_execution("Raw-to-Bronze", file_path, load_type, started, "FAILURE", error=exc)
        raise

In [0]:
symbols = discover_symbols(interval, load_type, year, month, day)
if not symbols:
    message = f"No uploaded {load_type} files for {year}-{month:02d}" + (f"-{day:02d}" if day else "")
    log_execution("Raw-to-Bronze", VOLUME_ROOT, load_type, utc_now(), "FAILURE" if require_files else "SKIPPED", error=message)
    if require_files:
        raise FileNotFoundError(f"{message} in {VOLUME_ROOT}; upload the files locally first")
    print("[SKIPPED]", message)
else:
    print(f"Discovered {len(symbols)} verified files already uploaded to {VOLUME_ROOT}")
    for sym in symbols:
        ingest_kline_file(sym, interval, load_type,
                          build_volume_path(sym, interval, load_type, year, month, day))

## exchangeInfo snapshot → Bronze
This is a separate feed uploaded from the same unchanged local extractor. A snapshot already ingested from the same file is skipped idempotently. The downstream Silver notebook uses the latest landed snapshot.

In [0]:
def ingest_exchange_info_snapshot(chosen_date=None):
    started = utc_now()
    folder = f"{VOLUME_ROOT}/exchange_info"
    try:
        entries = list_files(folder)
    except RuntimeError:
        entries = []
    files = sorted([f.path.replace("dbfs:", "") for f in entries
                    if re.fullmatch(r"exchangeInfo-\d{4}-\d{2}-\d{2}\.json", f.name)])
    if chosen_date:
        files = [p for p in files if p.endswith(f"exchangeInfo-{chosen_date}.json")]
    if not files:
        log_execution("Raw-to-Bronze", folder, "snapshot", started, "SKIPPED",
                      error="No exchangeInfo snapshot in Volume")
        print(f"[SKIPPED] No exchangeInfo JSON in {folder}")
        return
    path = files[-1]
    try:
        if (spark.catalog.tableExists(BRONZE_EXCHANGE_INFO_TABLE) and
            spark.table(BRONZE_EXCHANGE_INFO_TABLE)
                 .filter(F.col("source_file") == path).limit(1).count()):
            log_execution("Raw-to-Bronze", path, "snapshot", started, "SKIPPED",
                          error="Snapshot already ingested")
            return
        source = (spark.read.schema(RAW_TEXT_SCHEMA).text(path, wholetext=True)
                  .select(F.col("value").alias("raw_json"))
                  .withColumn("source_file", F.lit(path))
                  .withColumn("load_timestamp", F.current_timestamp()))
        if not source.limit(1).count():
            raise ValueError("Snapshot file is empty")
        (source.write.format("delta").mode("append")
               .option("mergeSchema", "true").saveAsTable(BRONZE_EXCHANGE_INFO_TABLE))
        log_execution("Raw-to-Bronze", path, "snapshot", started, "SUCCESS", inserted=1)
        print("[OK] exchangeInfo snapshot:", path)
    except Exception as exc:
        log_execution("Raw-to-Bronze", path, "snapshot", started, "FAILURE", error=exc)
        raise

ingest_exchange_info_snapshot(snapshot_date)

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/cryptolens/bronze_raw/"))